# **03 - Patient-Level Summaries**

In [24]:
import pandas as pd

df = pd.read_parquet("../data/processed/apc_clean.parquet")

In [3]:
# Sort for readmission calculation
df = df.sort_values(["patient_id", "adm"])

# Calculate 30-day readmission flag
df["next_adm"] = df.groupby("patient_id")["adm"].shift(-1)
df["readmit_30d"] = (df["next_adm"] - df["dis"]).dt.days.between(1, 30)

# Aggregate to patient-level
patients = (
    df.groupby("patient_id")
      .agg(
          n_spells=("spell_id", "nunique"),
          mean_los=("los_days", "mean"),
          pct_emerg=("any_emerg", "mean"),
          readmit_30d=("readmit_30d", "max"),
          imd_quintile=("imd_quintile", "first"),
          sex=("sex", "first"),
          age=("age", "mean"),
          ethnicity_group=("ethnicity_group", "first"),
          respiratory_group_mode=("respiratory_group", lambda x: x.mode()[0] if not x.mode().empty else "Unknown")
      )
      .reset_index()
)


In [4]:
print(f"Unique patients: {patients.shape[0]:,}")
print(f"Mean admissions per patient: {patients['n_spells'].mean():.2f}")
patients["readmit_30d"].value_counts(normalize=True)

Unique patients: 128,373
Mean admissions per patient: 1.13


readmit_30d
False    0.979396
True     0.020604
Name: proportion, dtype: float64

In [ ]:
# Validation checks

# Missingness
print(patients.isna().mean().sort_values(ascending=False))

# LOS sanity
print(patients["mean_los"].describe())

# Check for outliers in n_spells
print(patients["n_spells"].value_counts().head(10))

# Confirm demographics coverage
print(patients["ethnicity_group"].value_counts(normalize=True))
print(patients["imd_quintile"].value_counts(normalize=True).sort_index())


imd_quintile              0.017589
patient_id                0.000000
n_spells                  0.000000
mean_los                  0.000000
pct_emerg                 0.000000
readmit_30d               0.000000
sex                       0.000000
age                       0.000000
ethnicity_group           0.000000
respiratory_group_mode    0.000000
dtype: float64
count    128373.000000
mean         85.795511
std          89.756915
min           1.000000
25%           6.000000
50%          54.000000
75%         143.000000
max         696.000000
Name: mean_los, dtype: float64
n_spells
1     116715
2       8902
3       1672
4        520
5        224
6        120
7         72
8         66
9         37
10        23
Name: count, dtype: int64
ethnicity_group
White                     0.732039
Not stated                0.096321
Asian or Asian British    0.059031
Not known                 0.045617
Black or Black British    0.028550
Other Ethnic Groups       0.024740
Mixed                     0.0

In [25]:
df["los_days"].describe(percentiles=[0.9, 0.95, 0.99])

count    145305.000000
mean         87.206861
std          92.686794
min           1.000000
50%          53.000000
90%         236.000000
95%         277.000000
99%         328.000000
max         696.000000
Name: los_days, dtype: float64

In [26]:
(df["los_days"] > 180).mean()

0.1911083582808575

In [27]:
df = df[df["los_days"].between(1, 365)]

In [28]:
df["los_days"].describe()

count    145246.000000
mean         87.075169
std          92.463923
min           1.000000
25%           2.000000
50%          53.000000
75%         149.000000
max         365.000000
Name: los_days, dtype: float64

In [29]:
df["los_days"] = df["los_days"] / 10

In [30]:
df["los_days"].describe(percentiles=[0.9, 0.95, 0.99])

count    145246.000000
mean          8.707517
std           9.246392
min           0.100000
50%           5.300000
90%          23.600000
95%          27.700000
99%          32.700000
max          36.500000
Name: los_days, dtype: float64